# PFO Local Earthquake Database Construction

---
Remove events and compile entire dataset with dimensions [events, elements, temporal samples, components]

In [1]:
%%time
import shutil, pickle
#-----------------------------------------------------------------------------------------------------------------#
import numpy as np
import pandas as pd
#-----------------------------------------------------------------------------------------------------------------#
from obspy import read
from pathlib import Path
from collections import Counter, defaultdict

CPU times: user 555 ms, sys: 356 ms, total: 911 ms
Wall time: 3.62 s


## Sensor Quality Check

---
Checking how often sensors are down

In [4]:
%%time
mseed_dir = Path("/Path/to/waveforms") # downloaded from FDSN (not waveforms.npy)
components_to_check = ["BHZ", "BHN", "BHE"]
required_components = set(components_to_check)
expected_sensor_dim = 13

# expected_stations = {"BPH01", "BPH02", ...}
expected_stations = {f"BPH{i:02d}" for i in range(1, 14)}

# ---------- Pass 1: collect per-event station -> present components ----------
event_station_components = {}   # evid -> {station: set(components_present)}
station_event_presence = Counter()

mseed_files = sorted(mseed_dir.glob("*.mseed"))
for mseed_file in mseed_files:
    evid = mseed_file.stem.replace("evid_", "")
    st_all = read(str(mseed_file))

    sta_to_comps = defaultdict(set)
    for tr in st_all:
        ch = tr.stats.channel
        if ch in required_components:
            sta_to_comps[tr.stats.station].add(ch)

    event_station_components[evid] = dict(sta_to_comps)

    # Count each station at most once per event
    station_event_presence.update(set(sta_to_comps.keys()))

n_events = len(event_station_components)

# ---------- Pass 2: station-level missing counts ----------
missing_station_counts = Counter()                 # station -> #events where >=1 component missing
missing_component_counts = defaultdict(Counter)    # station -> per-component missing counts
events_with_any_missing_station = 0

for evid, sta_to_comps in event_station_components.items():
    missing_any_this_event = False

    for sta in expected_stations:
        present = sta_to_comps.get(sta, set())
        missing_comps = required_components - present

        # Count ONCE per station per event if anything missing
        if missing_comps:
            missing_station_counts[sta] += 1
            missing_any_this_event = True
            for comp in missing_comps:
                missing_component_counts[sta][comp] += 1

    if missing_any_this_event:
        events_with_any_missing_station += 1

# ---------- Summary ----------
print(f"Total events scanned: {n_events}")
print(f"Expected stations ({len(expected_stations)}): {sorted(expected_stations)}")
print(f"Events with >=1 missing station: {events_with_any_missing_station} / {n_events}\n")

rows = []
for sta in sorted(expected_stations):
    miss_n = missing_station_counts[sta]
    row = {
        "station": sta,
        "missing_event_count": miss_n,
        "missing_event_percent": 100.0 * miss_n / n_events if n_events else 0.0,
    }
    for comp in components_to_check:
        row[f"missing_{comp}_count"] = missing_component_counts[sta][comp]
    rows.append(row)

df_missing = pd.DataFrame(rows).sort_values("missing_event_count", ascending=False)
print(df_missing.to_string(index=False))

Total events scanned: 37151
Expected stations (13): ['BPH01', 'BPH02', 'BPH03', 'BPH04', 'BPH05', 'BPH06', 'BPH07', 'BPH08', 'BPH09', 'BPH10', 'BPH11', 'BPH12', 'BPH13']
Events with >=1 missing station: 12205 / 37151

station  missing_event_count  missing_event_percent  missing_BHZ_count  missing_BHN_count  missing_BHE_count
  BPH05                11100              29.878065              11100              11100              11100
  BPH04                 3751              10.096633               3751               3751               3751
  BPH12                 3508               9.442545               3508               3508               3508
  BPH07                 3007               8.093995               3007               3007               3007
  BPH02                 1706               4.592070               1706               1706               1706
  BPH13                 1480               3.983742               1480               1480               1480
  BPH03            

#### Looks like we should remove BPH05, BPH04, and BPH12

---
automated check

In [7]:
%%time
# Checking sensor dimension, temporal dimension, and zero amplitude - creating automated removal excel spreadsheet 
mseed_dir = Path("/Path/to/waveforms") # downloaded from FDSN (not waveforms.npy)
out_xlsx = Path("/Local_Catalog/Remove_EVID_automatically.xlsx")

components_to_check = ["BHZ", "BHN", "BHE"]
stations_to_remove = {"BPH04", "BPH05", "BPH12"} # not removing BPH07 for now to see how many events I still have

accepted_sensor_dim = 10
accepted_time_dims = {2400, 2401}
zero_fraction_threshold = 0.50  # >50% zeros => remove event

kept_event_ids = []
removed_event_ids = []
removed_reasons = {}

for mseed_file in sorted(mseed_dir.glob("*.mseed")):
    evid = mseed_file.stem.replace("evid_", "")

    try:
        st_all = read(str(mseed_file))

        for comp in components_to_check:
            st_comp = st_all.select(channel=comp).copy()

            # 1) Remove permanently bad stations FIRST
            for tr in list(st_comp):
                if tr.stats.station in stations_to_remove:
                    st_comp.remove(tr)

            if len(st_comp) == 0:
                raise ValueError(f"{comp}: no traces after removing bad stations")

            # Merge fragments where possible
            try:
                st_comp = st_comp.merge(method=1, fill_value=None)
            except Exception:
                st_comp = st_comp.split().merge(method=1, fill_value=None)

            # 2) Sensor dimension check (unique stations)
            stations_present = {tr.stats.station for tr in st_comp}
            sensor_dim = len(stations_present)
            if sensor_dim != accepted_sensor_dim:
                raise ValueError(f"{comp}: sensor_dim={sensor_dim} (expected {accepted_sensor_dim})")

            # 3) Temporal dimension check (2400 or 2401 only)
            bad_lengths = [tr.stats.npts for tr in st_comp if tr.stats.npts not in accepted_time_dims]
            if bad_lengths:
                raise ValueError(f"{comp}: invalid time_dims={sorted(set(bad_lengths))}")

            # Keep compatibility logic: treat 2401 as 2400
            final_lengths = [2400 if tr.stats.npts == 2401 else tr.stats.npts for tr in st_comp]
            if any(n != 2400 for n in final_lengths):
                raise ValueError(f"{comp}: final time_dims={sorted(set(final_lengths))}")

            # 4) Zero-amplitude fraction check (>10%)
            for tr in st_comp:
                data = tr.data
                if np.ma.isMaskedArray(data):
                    vals = np.asarray(data.filled(np.nan), dtype=float)
                else:
                    vals = np.asarray(data, dtype=float)

                vals = vals[np.isfinite(vals)]
                if vals.size == 0:
                    raise ValueError(f"{comp}: all-NaN/invalid samples on {tr.stats.station}")

                zero_frac = np.isclose(vals, 0.0).mean()
                if zero_frac > zero_fraction_threshold:
                    raise ValueError(
                        f"{comp}: {tr.stats.station} has {100*zero_frac:.1f}% zeros "
                        f"(>{100*zero_fraction_threshold:.0f}%)"
                    )

        kept_event_ids.append(evid)

    except Exception as e:
        removed_reasons[evid] = str(e)
        print(f"Removed {evid}: {e}")
        removed_event_ids.append(evid)

removed_event_ids = sorted(set(removed_event_ids), key=int)

# Save only EVID column
pd.DataFrame({"EVID": removed_event_ids}).to_excel(out_xlsx, index=False)

print(f"\nKept {len(kept_event_ids)} events")
print(f"Removed {len(removed_event_ids)} events")
print(f"Saved removed EVID list to: {out_xlsx}")

Removed 15535761: BHZ: sensor_dim=9 (expected 10)
Removed 15535769: BHZ: sensor_dim=9 (expected 10)
Removed 15535817: BHZ: sensor_dim=9 (expected 10)
Removed 15535841: BHZ: sensor_dim=9 (expected 10)
Removed 15535977: BHZ: sensor_dim=9 (expected 10)
Removed 15536033: BHZ: sensor_dim=9 (expected 10)
Removed 15536065: BHZ: sensor_dim=9 (expected 10)
Removed 15536145: BHZ: sensor_dim=9 (expected 10)
Removed 15536233: BHZ: sensor_dim=9 (expected 10)
Removed 15536257: BHZ: sensor_dim=9 (expected 10)
Removed 15536337: BHZ: sensor_dim=9 (expected 10)
Removed 15536345: BHZ: sensor_dim=9 (expected 10)
Removed 15536369: BHZ: sensor_dim=9 (expected 10)
Removed 15536393: BHZ: sensor_dim=9 (expected 10)
Removed 15536449: BHZ: sensor_dim=9 (expected 10)
Removed 15536529: BHZ: sensor_dim=9 (expected 10)
Removed 15536545: BHZ: sensor_dim=9 (expected 10)
Removed 15536561: BHZ: sensor_dim=9 (expected 10)
Removed 15536577: BHZ: sensor_dim=9 (expected 10)
Removed 15536601: BHZ: sensor_dim=9 (expected 10)


Removed 37261568: BHZ: sensor_dim=9 (expected 10)
Removed 37261576: BHZ: sensor_dim=9 (expected 10)
Removed 37261592: BHZ: sensor_dim=9 (expected 10)
Removed 37261632: BHZ: sensor_dim=9 (expected 10)
Removed 37261664: BHZ: sensor_dim=9 (expected 10)
Removed 37261704: BHZ: sensor_dim=9 (expected 10)
Removed 37261712: BHZ: sensor_dim=9 (expected 10)
Removed 37261720: BHZ: sensor_dim=9 (expected 10)
Removed 37261744: BHZ: sensor_dim=9 (expected 10)
Removed 37280615: BHZ: sensor_dim=4 (expected 10)
Removed 37280639: BHZ: sensor_dim=5 (expected 10)
Removed 37280703: BHZ: sensor_dim=9 (expected 10)
Removed 37311464: BHZ: sensor_dim=9 (expected 10)
Removed 37311488: BHZ: sensor_dim=9 (expected 10)
Removed 37311504: BHZ: sensor_dim=9 (expected 10)
Removed 37311807: BHZ: sensor_dim=4 (expected 10)
Removed 37312104: BHZ: sensor_dim=9 (expected 10)
Removed 37312136: BHZ: invalid time_dims=[1073]
Removed 37312184: BHZ: sensor_dim=9 (expected 10)
Removed 37324460: BHZ: sensor_dim=9 (expected 10)
Re

Removed 38086512: BHZ: invalid time_dims=[160]
Removed 38086536: BHZ: invalid time_dims=[80]
Removed 38086912: BHZ: sensor_dim=9 (expected 10)
Removed 38086920: BHZ: sensor_dim=9 (expected 10)
Removed 38086952: BHZ: sensor_dim=9 (expected 10)
Removed 38086992: BHZ: sensor_dim=9 (expected 10)
Removed 38087008: BHZ: sensor_dim=9 (expected 10)
Removed 38087080: BHZ: sensor_dim=9 (expected 10)
Removed 38087144: BHZ: sensor_dim=9 (expected 10)
Removed 38087472: BHZ: sensor_dim=9 (expected 10)
Removed 38087568: BHZ: sensor_dim=9 (expected 10)
Removed 38087656: BHZ: sensor_dim=9 (expected 10)
Removed 38087968: BHZ: sensor_dim=9 (expected 10)
Removed 38087984: BHZ: sensor_dim=9 (expected 10)
Removed 38088104: BHZ: sensor_dim=9 (expected 10)
Removed 38088208: BHZ: sensor_dim=9 (expected 10)
Removed 38088328: BHZ: sensor_dim=9 (expected 10)
Removed 38088352: BHZ: sensor_dim=9 (expected 10)
Removed 38088408: BHZ: sensor_dim=9 (expected 10)
Removed 38088480: BHZ: sensor_dim=9 (expected 10)
Removed

Removed 38099368: BHZ: sensor_dim=9 (expected 10)
Removed 38099384: BHZ: sensor_dim=9 (expected 10)
Removed 38099392: BHZ: sensor_dim=9 (expected 10)
Removed 38099416: BHZ: sensor_dim=9 (expected 10)
Removed 38099432: BHZ: sensor_dim=9 (expected 10)
Removed 38099464: BHZ: sensor_dim=9 (expected 10)
Removed 38099552: BHZ: sensor_dim=9 (expected 10)
Removed 38099632: BHZ: sensor_dim=9 (expected 10)
Removed 38099736: BHZ: sensor_dim=9 (expected 10)
Removed 38099776: BHZ: sensor_dim=9 (expected 10)
Removed 38099792: BHZ: sensor_dim=9 (expected 10)
Removed 38099848: BHZ: sensor_dim=9 (expected 10)
Removed 38099856: BHZ: sensor_dim=9 (expected 10)
Removed 38099864: BHZ: sensor_dim=9 (expected 10)
Removed 38099896: BHZ: sensor_dim=9 (expected 10)
Removed 38099904: BHZ: sensor_dim=9 (expected 10)
Removed 38100024: BHZ: sensor_dim=9 (expected 10)
Removed 38100112: BHZ: sensor_dim=9 (expected 10)
Removed 38100232: BHZ: sensor_dim=9 (expected 10)
Removed 38100288: BHZ: sensor_dim=9 (expected 10)


Removed 38695802: BHZ: sensor_dim=9 (expected 10)
Removed 38695986: BHZ: sensor_dim=9 (expected 10)
Removed 38696138: BHZ: sensor_dim=9 (expected 10)
Removed 38696154: BHZ: sensor_dim=9 (expected 10)
Removed 38698567: BHZ: sensor_dim=9 (expected 10)
Removed 38699391: BHZ: sensor_dim=9 (expected 10)
Removed 38699447: BHZ: sensor_dim=9 (expected 10)
Removed 38700815: BHZ: sensor_dim=9 (expected 10)
Removed 38701791: BHZ: sensor_dim=9 (expected 10)
Removed 38702335: BHZ: sensor_dim=9 (expected 10)
Removed 38702351: BHZ: sensor_dim=9 (expected 10)
Removed 38702663: BHZ: sensor_dim=9 (expected 10)
Removed 38703367: BHZ: sensor_dim=9 (expected 10)
Removed 38914271: BHZ: invalid time_dims=[40]
Removed 38940056: BHZ: sensor_dim=9 (expected 10)
Removed 38942712: BHZ: sensor_dim=9 (expected 10)
Removed 38942872: BHZ: sensor_dim=9 (expected 10)
Removed 38943088: BHZ: sensor_dim=9 (expected 10)
Removed 38943664: BHZ: sensor_dim=9 (expected 10)
Removed 38944712: BHZ: sensor_dim=9 (expected 10)
Remo

...

Removed 41071640: BHZ: sensor_dim=9 (expected 10)
Removed 41071696: BHZ: sensor_dim=9 (expected 10)
Removed 41071768: BHZ: sensor_dim=9 (expected 10)
Removed 41071800: BHZ: sensor_dim=9 (expected 10)
Removed 41071864: BHZ: sensor_dim=9 (expected 10)
Removed 41072176: BHZ: sensor_dim=9 (expected 10)
Removed 41072624: BHZ: sensor_dim=9 (expected 10)
Removed 41072960: BHZ: sensor_dim=9 (expected 10)
Removed 41073016: BHZ: sensor_dim=9 (expected 10)
Removed 41073024: BHZ: sensor_dim=9 (expected 10)
Removed 41073032: BHZ: sensor_dim=9 (expected 10)
Removed 41073104: BHZ: sensor_dim=9 (expected 10)
Removed 41073184: BHZ: sensor_dim=9 (expected 10)
Removed 41073488: BHZ: sensor_dim=9 (expected 10)
Removed 41073632: BHZ: sensor_dim=9 (expected 10)
Removed 41073648: BHZ: sensor_dim=9 (expected 10)
Removed 41073696: BHZ: sensor_dim=9 (expected 10)
Removed 41073792: BHZ: sensor_dim=9 (expected 10)
Removed 41073800: BHZ: sensor_dim=9 (expected 10)
Removed 41073808: BHZ: sensor_dim=9 (expected 10)


Removed 41126159: BHZ: sensor_dim=3 (expected 10)
Removed 41126175: BHZ: sensor_dim=3 (expected 10)
Removed 41126191: BHZ: sensor_dim=3 (expected 10)
Removed 41126215: BHZ: sensor_dim=4 (expected 10)
Removed 41126247: BHZ: sensor_dim=4 (expected 10)
Removed 41126327: BHZ: sensor_dim=6 (expected 10)
Removed 41126335: BHZ: sensor_dim=6 (expected 10)
Removed 41126375: BHZ: sensor_dim=6 (expected 10)
Removed 41126463: BHZ: sensor_dim=8 (expected 10)
Removed 41126487: BHZ: sensor_dim=8 (expected 10)
Removed 41127095: BHZ: sensor_dim=9 (expected 10)
Removed 41127263: BHZ: sensor_dim=9 (expected 10)
Removed 41127359: BHZ: sensor_dim=9 (expected 10)
Removed 41127847: BHZ: sensor_dim=9 (expected 10)
Removed 41127975: BHZ: sensor_dim=9 (expected 10)
Removed 41128015: BHZ: sensor_dim=9 (expected 10)
Removed 41128079: BHZ: sensor_dim=9 (expected 10)
Removed 41128295: BHZ: sensor_dim=9 (expected 10)
Removed 41128311: BHZ: sensor_dim=9 (expected 10)
Removed 41128319: BHZ: sensor_dim=9 (expected 10)


Removed 41141999: BHZ: sensor_dim=9 (expected 10)
Removed 41142039: BHZ: sensor_dim=9 (expected 10)
Removed 41142071: BHZ: sensor_dim=9 (expected 10)
Removed 41142079: BHZ: sensor_dim=9 (expected 10)
Removed 41142128: BHZ: sensor_dim=9 (expected 10)
Removed 41142255: BHZ: sensor_dim=9 (expected 10)
Removed 41142327: BHZ: sensor_dim=9 (expected 10)
Removed 41142447: BHZ: sensor_dim=9 (expected 10)
Removed 41142535: BHZ: sensor_dim=9 (expected 10)
Removed 41142615: BHZ: sensor_dim=9 (expected 10)
Removed 41142632: BHZ: sensor_dim=9 (expected 10)
Removed 41142655: BHZ: sensor_dim=9 (expected 10)
Removed 41142664: BHZ: sensor_dim=9 (expected 10)
Removed 41142679: BHZ: sensor_dim=9 (expected 10)
Removed 41142695: BHZ: sensor_dim=9 (expected 10)
Removed 41142711: BHZ: sensor_dim=9 (expected 10)
Removed 41142719: BHZ: sensor_dim=9 (expected 10)
Removed 41142720: BHZ: sensor_dim=9 (expected 10)
Removed 41142759: BHZ: sensor_dim=9 (expected 10)
Removed 41142807: BHZ: sensor_dim=9 (expected 10)


Removed 41149207: BHZ: sensor_dim=9 (expected 10)
Removed 41149351: BHZ: sensor_dim=9 (expected 10)
Removed 41149399: BHZ: sensor_dim=9 (expected 10)
Removed 41149423: BHZ: sensor_dim=9 (expected 10)
Removed 41149519: BHZ: sensor_dim=9 (expected 10)
Removed 41149696: BHZ: sensor_dim=9 (expected 10)
Removed 41149839: BHZ: sensor_dim=9 (expected 10)
Removed 41149855: BHZ: sensor_dim=9 (expected 10)
Removed 41149863: BHZ: sensor_dim=9 (expected 10)
Removed 41149872: BHZ: sensor_dim=8 (expected 10)
Removed 41149992: BHZ: sensor_dim=9 (expected 10)
Removed 41150304: BHZ: sensor_dim=9 (expected 10)
Removed 41150456: BHZ: sensor_dim=9 (expected 10)
Removed 41150736: BHZ: sensor_dim=9 (expected 10)
Removed 41150760: BHZ: sensor_dim=8 (expected 10)
Removed 41151352: BHZ: sensor_dim=8 (expected 10)
Removed 41151424: BHZ: sensor_dim=9 (expected 10)
Removed 41151560: BHZ: sensor_dim=9 (expected 10)
Removed 41151576: BHZ: sensor_dim=9 (expected 10)
Removed 41151616: BHZ: sensor_dim=9 (expected 10)


Removed 41245000: BHZ: sensor_dim=8 (expected 10)
Removed 41245152: BHZ: sensor_dim=8 (expected 10)
Removed 41248064: BHZ: sensor_dim=8 (expected 10)
Removed 41248992: BHZ: sensor_dim=8 (expected 10)
Removed 41249160: BHZ: sensor_dim=8 (expected 10)
Removed 41249176: BHZ: sensor_dim=8 (expected 10)
Removed 41249200: BHZ: sensor_dim=8 (expected 10)
Removed 41249232: BHZ: sensor_dim=8 (expected 10)
Removed 41249664: BHZ: sensor_dim=8 (expected 10)
Removed 41250112: BHZ: sensor_dim=8 (expected 10)
Removed 41250584: BHZ: sensor_dim=8 (expected 10)
Removed 41250944: BHZ: sensor_dim=8 (expected 10)
Removed 41250984: BHZ: sensor_dim=8 (expected 10)
Removed 41251040: BHZ: sensor_dim=8 (expected 10)
Removed 41251048: BHZ: sensor_dim=8 (expected 10)
Removed 41251112: BHZ: sensor_dim=8 (expected 10)
Removed 41251328: BHZ: sensor_dim=8 (expected 10)
Removed 41251464: BHZ: sensor_dim=8 (expected 10)
Removed 41251480: BHZ: sensor_dim=8 (expected 10)
Removed 41251568: BHZ: sensor_dim=8 (expected 10)


## Compile Plots
---
For automatically removed sensors (saved in removed_automatically in SSD)

In [ ]:
%%time
# Inputs
remove_xlsx = Path("/Local_Catalog/Remove_EVID_automatically.xlsx") # spreadsheet with automatically removed events
catalog_runs = Path("/Path/to/catalog_runs") # this is where array processing figures are saved 

src_array = catalog_runs / "figures" / "array_data"
src_fk = catalog_runs / "figures" / "fk"

dst_array = catalog_runs / "removed_automatically" / "array_data"
dst_fk = catalog_runs / "removed_automatically" / "fk"
dst_array.mkdir(parents=True, exist_ok=True)
dst_fk.mkdir(parents=True, exist_ok=True)

# EVID list
df = pd.read_excel(remove_xlsx)
evids = (
    df["EVID"].dropna().astype(str).str.strip()
      .str.replace(r"\.0$", "", regex=True)
      .str.replace(r"^evid_", "", regex=True)
      .drop_duplicates()
)

copied = 0
missing = []

for evid in evids:
    slug = f"evid_{evid}"

    f1 = src_array / f"{slug}_array_data.png"
    f2 = src_fk / f"{slug}_fk.png"
    f3 = src_fk / f"{slug}_fk_families.png"

    found = False
    for src, dst_dir in [(f1, dst_array), (f2, dst_fk), (f3, dst_fk)]:
        if src.exists():
            shutil.copy(src, dst_dir / src.name)
            copied += 1
            found = True

    if not found:
        missing.append(evid)

print(f"EVIDs in spreadsheet: {len(evids)}")
print(f"Files copied: {copied}")
print(f"EVIDs with no matching figures: {len(missing)}")

---
For manually removed sensors (saved in removed_manually in SSD)

In [ ]:
%%time
# Inputs
remove_xlsx = Path("/Local_Catalog/Remove_EVID_manually.xlsx") # spreadsheet with manually removed events
catalog_runs = Path("/Path/to/catalog_runs") # this is where array processing figures are saved 

src_array = catalog_runs / "figures" / "array_data"
src_fk = catalog_runs / "figures" / "fk"

dst_array = catalog_runs / "removed_manually" / "array_data"
dst_fk = catalog_runs / "removed_manually" / "fk"
dst_array.mkdir(parents=True, exist_ok=True)
dst_fk.mkdir(parents=True, exist_ok=True)

# EVID list
df = pd.read_excel(remove_xlsx)
evids = (
    df["EVID"].dropna().astype(str).str.strip()
      .str.replace(r"\.0$", "", regex=True)
      .str.replace(r"^evid_", "", regex=True)
      .drop_duplicates()
)

copied = 0
missing = []

for evid in evids:
    slug = f"evid_{evid}"

    f1 = src_array / f"{slug}_array_data.png"
    f2 = src_fk / f"{slug}_fk.png"
    f3 = src_fk / f"{slug}_fk_families.png"

    found = False
    for src, dst_dir in [(f1, dst_array), (f2, dst_fk), (f3, dst_fk)]:
        if src.exists():
            shutil.copy(src, dst_dir / src.name)
            copied += 1
            found = True

    if not found:
        missing.append(evid)

print(f"EVIDs in spreadsheet: {len(evids)}")
print(f"Files copied: {copied}")
print(f"EVIDs with no matching figures: {len(missing)}")

## Compile Database

---
Aggregating waveforms (filtered 0.5-10 Hz) and the event's row from the local catalog (event_db[evid]["stream"] this is the waveform data and event_db[evid]["meta"] this is the metadata, for example event_db[evid]["meta"]["distance_km"])

In [ ]:
%%time
# ---------------- Paths ----------------
catalog_xlsx = Path("/Local_Catalog/PFO_Local_Catalog.xlsx")
removed_auto_xlsx = Path("/Local_Catalog/Remove_EVID_automatically.xlsx")
removed_manual_xlsx = Path("/Local_Catalog/Remove_EVID_manually.xlsx")
mseed_dir = Path("/Path/to/waveforms") # from FDSN (not waveforms.npy)
db_dir = Path("Database") # where it will save to (saves waveforms.npy, event_metadata.pkl, and dataset_info.pkl)
db_dir.mkdir(parents=True, exist_ok=True)

# ---------------- Config ----------------
components = ["BHZ", "BHN", "BHE"]
stations_to_remove = {"BPH04", "BPH05", "BPH12"}
station_order = [f"BPH{i:02d}" for i in range(1, 14) if f"BPH{i:02d}" not in stations_to_remove]  # 10 stations
accepted_npts = {2400, 2401}
target_npts = 2400

# ---------------- Helpers ----------------
def normalize_evid(series):
    return (
        series.dropna()
        .astype(str)
        .str.strip()
        .str.replace(r"^evid_", "", regex=True)
        .str.replace(r"\.0$", "", regex=True)
    )

# ---------------- Load removed EVID sets ----------------
removed_auto = set(normalize_evid(pd.read_excel(removed_auto_xlsx)["EVID"]))
removed_manual = set(normalize_evid(pd.read_excel(removed_manual_xlsx)["EVID"]))
removed_all = removed_auto | removed_manual

# ---------------- Load catalog and keep only non-removed ----------------
cat = pd.read_excel(catalog_xlsx).copy()
cat["EVID_norm"] = normalize_evid(cat["EVID"])
cat_kept = (
    cat[~cat["EVID_norm"].isin(removed_all)]
    .drop_duplicates(subset="EVID_norm")
    .set_index("EVID_norm")
)

# ---------------- Build waveform tensor + metadata ----------------
waveforms = []         # each entry shape: [n_comp, n_sta, n_samp]
meta_rows = []
missing_mseed = []
failed_events = []

for evid in cat_kept.index:
    mseed_file = mseed_dir / f"evid_{evid}.mseed"
    if not mseed_file.exists():
        missing_mseed.append(evid)
        continue

    try:
        st = read(str(mseed_file))

        # Remove bad stations
        st = st.copy()
        for tr in list(st):
            if tr.stats.station in stations_to_remove:
                st.remove(tr)

        # Merge first, then filter
        try:
            st = st.merge(method=1, fill_value=None)
        except Exception:
            st = st.split().merge(method=1, fill_value=None)

        # Preprocess
        st.detrend("demean")
        st.detrend("linear")
        st.taper(type="cosine", max_percentage=0.05)
        st.filter(
            "bandpass",
            freqmin=0.5,
            freqmax=10.0,
            corners=4,
            zerophase=True
        )

        event_arr = np.zeros((len(components), len(station_order), target_npts), dtype=np.float32)

        for ci, comp in enumerate(components):
            for si, sta in enumerate(station_order):
                tr_sel = st.select(channel=comp, station=sta)
                if len(tr_sel) != 1:
                    raise ValueError(f"Expected 1 trace for {sta}.{comp}, got {len(tr_sel)}")

                tr = tr_sel[0]
                npts = tr.stats.npts
                if abs(tr.stats.sampling_rate - 40.0) > 1e-6:
                    raise ValueError(f"{sta}.{comp} sampling rate is {tr.stats.sampling_rate}, expected 40.0")
                if npts not in accepted_npts:
                    raise ValueError(f"{sta}.{comp} has npts={npts}")
                if np.ma.isMaskedArray(tr.data):
                    raise ValueError(f"{sta}.{comp} contains masked/gapped data")
                    
                data = tr.data.astype(np.float32, copy=False)
                if npts == 2401:
                    data = data[:-1]  # enforce 2400
                if data.shape[0] != target_npts:
                    raise ValueError(f"{sta}.{comp} final npts={data.shape[0]} != {target_npts}")


                event_arr[ci, si, :] = data

        waveforms.append(event_arr)

        meta = cat_kept.loc[evid].to_dict()
        meta["EVID"] = evid
        meta["mseed_path"] = str(mseed_file)
        meta_rows.append(meta)

    except Exception as e:
        failed_events.append((evid, str(e)))

# Final arrays/dataframes
X = np.stack(waveforms, axis=0) if waveforms else np.empty((0, len(components), len(station_order), target_npts), dtype=np.float32)
meta_df = pd.DataFrame(meta_rows)

# ---------------- Save ----------------
np.save(db_dir / "waveforms.npy", X)  # shape: [N, C, S, T]
meta_df.to_pickle(db_dir / "event_meta.pkl")

with open(db_dir / "dataset_info.pkl", "wb") as f:
    pickle.dump(
        {
            "shape": X.shape,
            "dtype": str(X.dtype),
            "components": components,
            "station_order": station_order,
            "target_npts": target_npts,
            "missing_mseed": missing_mseed,
            "failed_events": failed_events,
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

print(f"Total catalog events: {len(cat)}")
print(f"Removed by lists: {len(removed_all)}")
print(f"Candidate kept events: {len(cat_kept)}")
print(f"Loaded events: {X.shape[0]}")
print(f"Missing mseed files: {len(missing_mseed)}")
print(f"Failed events: {len(failed_events)}")
print(f"Saved: {db_dir / 'waveforms.npy'}")
print(f"Saved: {db_dir / 'event_meta.pkl'}")
print(f"Saved: {db_dir / 'dataset_info.pkl'}")